# Llama-Instruct – Exploratory Data Analysis
Exploring Llama-Instruct's interoceptive word ratings against human ground-truth ratings.

**Scale:** 0 (not at all) → 5 (greatly evokes an internal bodily sensation)

> **Note:** Llama-Instruct responds with "0" for ~81 % of words — the expected score
> captures the residual probability mass on higher tokens and is the more informative metric.


## 1. Imports & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

from data_loading import load_data

sns.set_theme(style="whitegrid")
COLOR = "#4C72B0"
PREFIX = "llama_instruct"

## 2. Load Data

In [ ]:
df_all = load_data()

# Work with just the columns relevant to this model
df = df_all[["WORD", "human", "llama_instruct_response", "llama_instruct_expected", "llama_instruct_error"]].copy()
df = df.rename(columns={
    "llama_instruct_response": "response",
    "llama_instruct_expected": "expected",
    "llama_instruct_error":    "error",
})

print(f"Dataset: {len(df):,} words")
df.head()

## 3. Descriptive Statistics

### 3.1 Summary statistics

In [ ]:
desc = df[["human", "response", "expected"]].describe().T
desc.index = ["Human", f"Llama-Instruct Response", f"Llama-Instruct Expected Score"]
desc.round(3)

### 3.2 Skewness, kurtosis & median

In [ ]:
extra = pd.DataFrame([
    {
        "": label,
        "Skewness": df[col].skew(),
        "Kurtosis": df[col].kurt(),
        "Median":   df[col].median(),
    }
    for col, label in [
        ("human",    "Human"),
        ("response", "Llama-Instruct Response"),
        ("expected", "Llama-Instruct Expected Score"),
    ]
]).set_index("").round(3)

extra

### 3.3 Response value counts

In [ ]:
vc  = df["response"].value_counts().sort_index()
pct = (vc / len(df) * 100).round(1)
pd.DataFrame({"count": vc, "%": pct})

## 4. Distributions

### 4.1 Histogram + KDE: Human vs Expected Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (col, label, color) in zip(axes, [
    ("human",    "Human Ratings",             "#888888"),
    ("expected", "Llama-Instruct Expected Score",    COLOR),
]):
    ax.hist(df[col], bins=40, color=color, alpha=0.5, density=True, edgecolor="white")
    df[col].plot.kde(ax=ax, color=color, linewidth=2)
    ax.axvline(df[col].mean(),   color="black", linestyle="--", linewidth=1, label=f"Mean={df[col].mean():.2f}")
    ax.axvline(df[col].median(), color="black", linestyle=":",  linewidth=1, label=f"Median={df[col].median():.2f}")
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 3)
    ax.set_title(label)
    ax.set_xlabel("Score (0–5)")
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("Score Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.2 Discrete response bar chart

In [ ]:
vc  = df["response"].value_counts().sort_index()
pct = (vc / len(df) * 100)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(vc.index.astype(str), pct.values, color=COLOR, edgecolor="white")

for bar, p in zip(bars, pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{p:.1f}%", ha="center", va="bottom", fontsize=9)

ax.set_title("Llama-Instruct Discrete Response Distribution")
ax.set_xlabel("Response (0 = not at all, 5 = greatly)")
ax.set_ylabel("% of responses")
plt.tight_layout()
plt.show()

## 5. Correlation with Human Ratings

In [ ]:
pearson_r,  pearson_p  = stats.pearsonr(df["human"], df["expected"])
spearman_r, spearman_p = stats.spearmanr(df["human"], df["expected"])

print(f"Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.2e})")
print(f"Spearman ρ = {spearman_r:.4f}  (p = {spearman_p:.2e})")

### 5.1 Scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

hb = ax.hexbin(df["human"], df["expected"], gridsize=50,
               cmap="Blues", mincnt=1, linewidths=0.1)
plt.colorbar(hb, ax=ax, label="Word count")

m, b = np.polyfit(df["human"], df["expected"], 1)
x_line = np.linspace(0, 5, 200)
ax.plot(x_line, m * x_line + b, color=COLOR, linewidth=2, label=f"Fit (slope={m:.2f})")
ax.plot([0, 5], [0, 5], color="gray", linestyle="--", linewidth=1, label="Perfect agreement")

ax.set_xlabel("Human Rating")
ax.set_ylabel("Llama-Instruct Expected Score")
ax.set_title(f"Llama-Instruct vs Human  (r={pearson_r:.3f}, ρ={spearman_r:.3f})")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Error & Bias

In [ ]:
mae  = df["error"].abs().mean()
rmse = np.sqrt((df["error"] ** 2).mean())
bias = df["error"].mean()

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Bias: {bias:+.4f}  ({'over-rates' if bias > 0 else 'under-rates'})")

### 6.1 Bland-Altman plot

In [ ]:
mean_score = (df["human"] + df["expected"]) / 2
md_val = df["error"].mean()
sd_val = df["error"].std()
loa_hi = md_val + 1.96 * sd_val
loa_lo = md_val - 1.96 * sd_val

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(mean_score, df["error"], alpha=0.05, s=3, color=COLOR)
ax.axhline(md_val, color="black", linewidth=1.5, label=f"Bias = {md_val:+.3f}")
ax.axhline(loa_hi, color="red",   linewidth=1, linestyle="--", label=f"+1.96 SD = {loa_hi:.3f}")
ax.axhline(loa_lo, color="red",   linewidth=1, linestyle="--", label=f"-1.96 SD = {loa_lo:.3f}")
ax.axhline(0,      color="gray",  linewidth=0.8, linestyle=":")
ax.set_xlabel("Mean of Llama-Instruct and Human Score")
ax.set_ylabel("Llama-Instruct − Human")
ax.set_title("Bland-Altman Agreement Plot")
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Bias by human-rating bin

In [ ]:
df["human_bin"] = pd.cut(df["human"], bins=[0, 1, 2, 3, 4, 5],
                         include_lowest=True, labels=["0–1","1–2","2–3","3–4","4–5"])

binned = df.groupby("human_bin", observed=True)["error"].agg(["mean","std"]).reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(binned["human_bin"], binned["mean"], yerr=binned["std"],
       color=COLOR, alpha=0.7, capsize=4, error_kw={"elinewidth": 1})
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Human Rating Bin")
ax.set_ylabel("Mean Error (Llama-Instruct − Human)")
ax.set_title("Llama-Instruct Prediction Bias by Human-Rating Bin")
plt.tight_layout()
plt.show()

## 7. Clustering

In [ ]:
features = ["human", "response", "expected", "error"]
X = df[features].dropna()

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Clustering on {len(X):,} words with features: {features}")

### 7.1 Elbow method & silhouette scores

In [ ]:
k_range     = range(2, 11)
inertias    = []
silhouettes = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels, sample_size=5000, random_state=42))
    print(f"k={k}  inertia={km.inertia_:,.0f}  silhouette={silhouettes[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_range, inertias,    marker="o", color=COLOR)
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_range, silhouettes, marker="o", color=COLOR)
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")

plt.suptitle(f"Choosing k for K-Means (Llama-Instruct)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 7.2 Fit final model
Update `k` based on the plots above.

In [ ]:
k = 3  # <-- update based on plots above

km_final     = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_scaled)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

### 7.3 Cluster profiles

In [ ]:
profile = df.groupby("cluster")[features].mean().round(3)
profile["n_words"] = df.groupby("cluster").size()
profile

### 7.4 PCA visualisation

In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")

palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974"]

fig, ax = plt.subplots(figsize=(8, 6))
for c in sorted(df["cluster"].unique()):
    mask = df["cluster"] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               alpha=0.3, s=5, label=f"Cluster {c}", color=palette[c])

ax.set_title(f"K-Means Clusters (k={k}) — PCA projection (Llama-Instruct)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

### 7.5 Example words per cluster

In [ ]:
for c in sorted(df["cluster"].unique()):
    sample = df[df["cluster"] == c].nlargest(10, "human")[["WORD","human","expected","error"]]
    print(f"\n── Cluster {c} (top 10 by human rating) ──")
    print(sample.round(3).to_string(index=False))

## 8. Interesting Words

### 8.1 Largest disagreements

In [ ]:
df["abs_error"] = df["error"].abs()
df.nlargest(20, "abs_error")[["WORD","human","expected","error"]].round(3)

### 8.2 Top human-rated words

In [ ]:
df.nlargest(20, "human")[["WORD","human","expected","error"]].round(3)